In [1]:
# Notebook 3 — Embeddings, feature fusion, and robust stratified splits

import os
import json
import math
import random
from datetime import datetime

import numpy as np
import pandas as pd
from scipy import sparse

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# IO paths
PROCESSED_DIR = "data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Embedding config
EMBED_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"  # multilingual for Pakistan context

print("Config ready.")

Config ready.


In [9]:
# Expect descriptions_clean.csv from Notebook 1
cleaned_path = os.path.join(PROCESSED_DIR, "descriptions_clean.csv")
if not os.path.exists(cleaned_path):
    raise FileNotFoundError("Missing data/processed/descriptions_clean.csv. Run 02_1_data_audit_schema_alignment first.")

df = pd.read_csv(cleaned_path)

# Required columns
required_cols = ["description_clean", "osm_tag_key", "osm_tag_value"]
for c in required_cols:
    if c not in df.columns:
        raise ValueError(f"Missing required column: {c}")

# Create canonical label string: key=value
df["label_str"] = (df["osm_tag_key"].astype(str).str.strip() + "=" + df["osm_tag_value"].astype(str).str.strip())

# Map to ids
labels_sorted = sorted(df["label_str"].unique())
label_to_id = {lbl: i for i, lbl in enumerate(labels_sorted)}
id_to_label = {str(i): lbl for lbl, i in label_to_id.items()}

df["label_id"] = df["label_str"].map(label_to_id).astype(int)

print(f"Loaded {len(df)} rows with {len(labels_sorted)} classes.")
print("Class sample:", labels_sorted[:5])

ValueError: Missing required column: description_clean